In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# Datasets and Dataloaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [5]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

# Build CNN

In [15]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x  = x.view(x.size(0),-1)
        x = self.fc_layers(x)

        return x

In [16]:
model = CNN()

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [19]:
# Training the CNN

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss = 0.21582157993712997
epoch=2/10 & loss = 0.16124701125742605
epoch=3/10 & loss = 0.13153866149932908
epoch=4/10 & loss = 0.11051219965443206
epoch=5/10 & loss = 0.09796611401026169
epoch=6/10 & loss = 0.09561793901659835
epoch=7/10 & loss = 0.07669592483768292
epoch=8/10 & loss = 0.07780544862018356
epoch=9/10 & loss = 0.07401969512600613
epoch=10/10 & loss = 0.0736998433654811


In [20]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}%")
    

Accuracy = 74.28%
